# 📄 Processamento de Certidões de Óbito com olmOCR
Notebook completo para processar imagens de certidões de óbito usando o modelo `allenai/olmOCR-7B-0225-preview` no Google Colab (GPU gratuita).

## Passos prévios:
1. Crie uma conta no [Hugging Face](https://huggingface.co)
2. Solicite acesso ao modelo [allenai/olmOCR-7B-0225-preview](https://huggingface.co/allenai/olmOCR-7B-0225-preview)
3. Gere um token de acesso em [Hugging Face Tokens](https://huggingface.co/settings/tokens)
4. Comprima a pasta `full_images` do seu projeto num ficheiro `.zip` no seu computador local

In [ ]:
# Passo 1: Instalar dependências necessárias
!pip install -q transformers accelerate pillow torch huggingface_hub bitsandbytes

In [ ]:
# Passo 2: Login no Hugging Face (cole o token gerado anteriormente)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Passo 3: Carregar imagens - Faça upload do ficheiro zip da pasta full_images
from google.colab import files
import zipfile
import os

print("Por favor, faça upload do ficheiro zip da pasta full_images...")
uploaded = files.upload()

# Descomprimir o ficheiro
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        zip_path = filename
        extract_dir = 'full_images'
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Imagens extraídas para: {extract_dir}")
        break

In [ ]:
# Passo 4: Verificar imagens carregadas
import glob

image_dir = 'full_images'
image_files = glob.glob(os.path.join(image_dir, '**/*.tiff'), recursive=True) + \
               glob.glob(os.path.join(image_dir, '**/*.png'), recursive=True)

print(f"Total de imagens encontradas: {len(image_files)}")
if image_files:
    print(f"Exemplo de imagem: {image_files[0]}")

In [ ]:
# Passo 5: Carregar modelo olmOCR com quantização 4-bit (poupa memória GPU)
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig

model_name = "allenai/olmOCR-7B-0225-preview"
print(f"A carregar modelo: {model_name}")

# Configuração de quantização 4-bit para caber na GPU do Colab
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Carregar processador e modelo
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Modelo carregado com sucesso!")

In [ ]:
# Passo 6: Função para processar imagens com olmOCR
from PIL import Image
import json

def process_image(image_path):
    """Processa uma imagem de certidão de óbito e extrai informação estruturada"""
    try:
        # Abrir imagem (tratar TIFF e PNG)
        img = Image.open(image_path).convert("RGB")
        
        # Prompt otimizado para certidões de óbito portuguesas
        prompt = """Extract all information from this Portuguese death certificate image. 
        Return ONLY a JSON object with these fields (use null if not found):
        - numero_assento
        - nome
        - data_obito
        - idade
        - sexo
        - estado_civil
        - causa_morte
        - local_obito
        - residencia"""
        
        # Preparar inputs para o modelo
        inputs = processor(text=prompt, images=img, return_tensors="pt").to(model.device)
        
        # Gerar resultado
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512)
        
        # Decodificar resultado
        result = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        
        # Limpar possível texto extra
        if "```json" in result:
            result = result.split("```json")[1].split("```")[0].strip()
        
        return json.loads(result) if result.strip().startswith('{') else {"raw_text": result}
    
    except Exception as e:
        return {"error": str(e), "image": image_path}

print("Função de processamento definida.")

In [ ]:
# Passo 7: Testar com 1 imagem primeiro
if image_files:
    test_image = image_files[0]
    print(f"A processar imagem de teste: {test_image}")
    test_result = process_image(test_image)
    print("Resultado do teste:")
    print(json.dumps(test_result, indent=2, ensure_ascii=False))
else:
    print("Nenhuma imagem encontrada.")

In [ ]:
# Passo 8: Processar todas as imagens (ou um subconjunto)
import time

# Configurar número de imagens a processar (None para todas)
MAX_IMAGES = 10  # Altere para None para processar todas

images_to_process = image_files[:MAX_IMAGES] if MAX_IMAGES else image_files
print(f"A processar {len(images_to_process)} imagens...")

results = []
for i, img_path in enumerate(images_to_process):
    print(f"Progresso: {i+1}/{len(images_to_process)} - {os.path.basename(img_path)}")
    result = process_image(img_path)
    results.append({
        "image": os.path.basename(img_path),
        "image_path": img_path,
        "data": result
    })
    time.sleep(0.5)  # Pequena pausa para evitar sobrecarga

print("Processamento concluído!")

In [ ]:
# Passo 9: Guardar resultados e fazer download
import json
from google.colab import files

# Guardar resultados em JSON
output_file = "resultados_obitos.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Resultados guardados em: {output_file}")
print(f"Total de registos processados: {len(results)}")

# Fazer download do ficheiro
files.download(output_file)

## Próximos Passos
1. Os resultados estão no ficheiro `resultados_obitos.json` no seu computador
2. Pode ajustar o `MAX_IMAGES` para processar mais imagens (todas demoram ~1-2 min por imagem)
3. Para processar todas as 4000+ imagens, recomenda-se usar o Colab Pro (GPU mais rápida) ou dividir o processamento em lotes